# Pauli Algebra 07: TFIM Dynamical Quench Application

This notebook is a practical dynamics example for the transverse-field Ising model (TFIM). It uses the Pauli-algebra Lie-closure route as the main simulation path and cross-checks against the dense and analytic/free-fermion utilities in `quantum_simulation.dynamic`.


## What you will learn

1. How to build the TFIM Hamiltonian in the Pauli backend with the same convention used by `dynamic.py`.
2. How to simulate average longitudinal magnetization after a field quench.
3. How to cross-check a small system against `dynamic_of_observables`.
4. How to cross-check a length-10 system against `tf_xy_chain_mz_instant_quench`.


In [17]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [18]:
import warnings

import numpy as np
import torch

from quantum_simulation.dynamic import (
    dynamic_of_observables,
    tf_xy_chain_mz_instant_quench,
)
from quantum_simulation.pauli_algebra import (
    PauliHamiltonian,
    PauliSum,
    SparseKet,
    pauli_string_from_sites,
)

warnings.filterwarnings(
    "ignore",
    message="The use of `x.T` on tensors of dimension other than 2*",
    category=UserWarning,
)

np.set_printoptions(precision=6, suppress=True)
device = "cpu"
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## TFIM convention used here

The Pauli backend builds

$$H = -J \sum_i S_i^x S_{i+1}^x - h \sum_i S_i^z,$$

so in Pauli matrices this is

$$H = -\frac{J}{4}\sum_i X_i X_{i+1} - \frac{h}{2}\sum_i Z_i.$$

This matches the normalization inside `tf_xy_chain_mz_instant_quench`, where the function divides `jx` by 4 and the transverse field by 2. The magnetization observable below is the average Pauli-$Z$ magnetization, `M_z = (1/L) sum_i Z_i`, so the all-up computational basis state has `M_z = 1`.


In [19]:
def build_tfim_model(lattice_length, *, j_coupling, h_field, pbc=True):
    model = PauliHamiltonian(lattice_length=lattice_length)
    num_bonds = lattice_length if pbc else lattice_length - 1
    for i in range(num_bonds):
        model.add_spin_term(-j_coupling, ["S_x", "S_x"], [i, (i + 1) % lattice_length])
    for i in range(lattice_length):
        model.add_spin_term(-h_field, ["S_z"], [i])
    return model


def average_mz_operator(lattice_length):
    return PauliSum([
        pauli_string_from_sites("Z", [i], lattice_length, coefficient=1.0 / lattice_length)
        for i in range(lattice_length)
    ]).simplify()


def all_up_state_vector(lattice_length, *, device="cpu"):
    psi = torch.zeros(2**lattice_length, dtype=torch.complex128, device=device)
    psi[0] = 1.0 + 0.0j
    return psi


def pauli_lie_mz_curve(model, observable, times, *, device="cpu"):
    state = SparseKet.basis(model.lattice_length, 0, device=device)
    basis = model.lie_closure_basis([observable], atol=1e-10, rtol=1e-10, max_iter=100)
    values = []
    for time in times:
        value, _, _, _ = model.expectation(
            basis,
            observable,
            state,
            time=float(time),
            device=device,
        )
        values.append(float(np.real_if_close(value)))
    return np.array(values), basis


def print_curve_table(times, left, right, *, left_name="Pauli", right_name="reference"):
    print(f"{'t':>7}  {left_name:>14}  {right_name:>14}  {'abs diff':>10}")
    for t, a, b in zip(times, left, right):
        print(f"{t:7.3f}  {a:14.10f}  {b:14.10f}  {abs(a - b):10.3e}")


## Quench setup

The Pauli simulation starts from the all-up state `|00...0>`, then evolves with the post-quench Hamiltonian. The analytic/free-fermion helper in `dynamic.py` starts from the ground state of the pre-quench field. We use a very large positive `h_initial` to approximate the same all-up initial state.


In [20]:
J = 1.0
h_initial = 1.0e6
h_final = 0.7
pbc = True
times = np.linspace(0.0, 2.0, 9)

print("times =", times)
print(f"J={J}, h_initial={h_initial:.1e}, h_final={h_final}, pbc={pbc}")


times = [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.  ]
J=1.0, h_initial=1.0e+06, h_final=0.7, pbc=True


## Small system: cross-check with `dynamic_of_observables`

For `L=4`, dense exact evolution is cheap. We compare the Pauli Lie-closure result against `dynamic_of_observables`, using the same dense Hamiltonian and observable matrices produced by the Pauli backend.


In [24]:
L_small = 4
model_small = build_tfim_model(L_small, j_coupling=J, h_field=h_final, pbc=pbc)
observable_small = average_mz_operator(L_small)

pauli_small, basis_small = pauli_lie_mz_curve(model_small, observable_small, times, device=device)

H_small = torch.as_tensor(model_small.to_matrix(), dtype=dtype, device=device)
O_small = torch.as_tensor(observable_small.to_matrix(), dtype=dtype, device=device)
psi0_small = all_up_state_vector(L_small, device=device)

dense_small = dynamic_of_observables(
    list(times),
    O_small,
    psi0_small,
    H_small,
).squeeze(-1).detach().cpu().numpy()

print("small-system Lie basis dimension:", len(basis_small))
print_curve_table(times, pauli_small, dense_small, left_name="Pauli Lie", right_name="dynamic.py")
max_small_error = np.max(np.abs(pauli_small - dense_small))
print("max small-system error:", max_small_error)
assert max_small_error < 1e-10


small-system Lie basis dimension: 11
      t       Pauli Lie      dynamic.py    abs diff
  0.000    1.0000000000    1.0000000000   6.661e-16
  0.250    0.9846137468    0.9846137468   7.772e-16
  0.500    0.9412194652    0.9412194652   7.772e-16
  0.750    0.8773924677    0.8773924677   3.331e-16
  1.000    0.8035736793    0.8035736793   4.441e-16
  1.250    0.7304622381    0.7304622381   7.772e-16
  1.500    0.6664978868    0.6664978868   4.441e-16
  1.750    0.6161479015    0.6161479015   6.661e-16
  2.000    0.5794636301    0.5794636301   1.332e-15
max small-system error: 1.3322676295501878e-15


## Large system: length 10 cross-check with `tf_xy_chain_mz_instant_quench`

For `L=10`, dense Hilbert-space evolution already uses a `1024 x 1024` Hamiltonian. The Pauli Lie-closure route stays in a much smaller operator subspace for this TFIM observable. We cross-check it against the analytic/free-fermion TF-XY chain helper with `jy=0`, which is the transverse-field Ising limit.


In [25]:
L_large = 10
model_large = build_tfim_model(L_large, j_coupling=J, h_field=h_final, pbc=pbc)
observable_large = average_mz_operator(L_large)

pauli_large, basis_large = pauli_lie_mz_curve(model_large, observable_large, times, device=device)
free_fermion_large = tf_xy_chain_mz_instant_quench(
    list(times),
    lattice_length=L_large,
    jx=J,
    jy=0.0,
    h_initial=h_initial,
    h_final=h_final,
)

print("large-system qubits:", L_large)
print("Hamiltonian Pauli-term count:", model_large.num_terms())
print("large-system Lie basis dimension:", len(basis_large))
print_curve_table(times, pauli_large, free_fermion_large, left_name="Pauli Lie", right_name="TF-XY formula")
max_large_error = np.max(np.abs(pauli_large - free_fermion_large))
print("max length-10 error:", max_large_error)
assert max_large_error < 1e-6


large-system qubits: 10
Hamiltonian Pauli-term count: 20
large-system Lie basis dimension: 29
      t       Pauli Lie   TF-XY formula    abs diff
  0.000    1.0000000000    1.0000000000   0.000e+00
  0.250    0.9846140758    0.9846140866   1.074e-08
  0.500    0.9412399042    0.9412399449   4.070e-08
  0.750    0.8776139474    0.8776140310   8.361e-08
  1.000    0.8047335820    0.8047337127   1.307e-07
  1.250    0.7345010442    0.7345012170   1.728e-07
  1.500    0.6772709498    0.6772711523   2.024e-07
  1.750    0.6398761786    0.6398763938   2.152e-07
  2.000    0.6245681337    0.6245683448   2.111e-07
max length-10 error: 2.152395468524304e-07


## Reading the result

The small-system comparison verifies the Pauli backend against dense exact Schrödinger evolution. The length-10 comparison verifies the same Pauli Lie-closure workflow against the specialized free-fermion TFIM/XY quench formula. The small remaining difference in the length-10 check comes from representing the ideal all-up initial state in the analytic formula with a finite but very large `h_initial`.
